# DISTILBERT FINE-TUNING

In [3]:
from collections import Counter
import json

def cargar_datos_entrenamiento(ruta_archivo):
    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    X = []
    y = []
    ids = []

    for key, value in data.items():
        tweet_text = value['tweet']
        votos = value['labels_task1']
        
        conteo = Counter(votos)
        label_mayoritaria, frecuencia = conteo.most_common(1)[0]
        total_votos = len(votos)
        if frecuencia <= total_votos / 2:
            continue 
            
        X.append(tweet_text)
        y.append(label_mayoritaria)
        ids.append(value['id_EXIST'])

    return X, y, ids



ruta = 'dataset_task1_exist2025/training.json' 
try:
    X_train, y_train, ids_train = cargar_datos_entrenamiento(ruta)
    
    print(f"Procesados {len(X_train)} tweets correctamente.")
    print(f"Ejemplo - Texto: {X_train[0][:50]}...")
    print(f"Ejemplo - Label: {y_train[0]}")

except FileNotFoundError:
    print("Error: No se encontró el archivo training.json. Revisa la ruta.")

Procesados 6064 tweets correctamente.
Ejemplo - Texto: @TheChiflis Ignora al otro, es un capullo.El probl...
Ejemplo - Label: YES


In [5]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType
import numpy as np
import evaluate

model_id = "distilbert-base-multilingual-cased" 

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Formato Hugging Face
label_map = {"YES": 1, "NO": 0}
train_dict = {
    "text": X_train, 
    "label": [label_map[l] for l in y_train]
}
ds = Dataset.from_dict(train_dict)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=True, max_length=128)

tokenized_ds = ds.map(preprocess_function, batched=True)

c:\Users\franm\Desktop\SEXISM_TWEETS\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\franm\.cache\huggingface\hub\models--distilbert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 6064/6064 [00:00<00:00, 17173.01 examples/s]


In [7]:
import optuna
from sklearn.model_selection import StratifiedKFold
import numpy as np

metric = evaluate.load("f1")
# Split 80/20 antes de entrenar
split_ds = tokenized_ds.train_test_split(test_size=0.2, seed=42)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"]
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels, average="macro")

def model_init(trial=None):
    base_model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
    return get_peft_model(base_model, peft_config)

# Espacio de búsqueda con Optuna
def my_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 5e-4, log=True),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 4),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [16, 32]),
    }


trainer_hpo = Trainer(
    model_init=model_init,
    args=TrainingArguments(
        output_dir="./hpo_results",
        eval_strategy="epoch",
        save_strategy="no",
        disable_tqdm=False, 
        warmup_ratio=0.1,
        weight_decay=0.01,
        logging_steps=10,
        fp16=True if torch.cuda.is_available() else False
    ),
    train_dataset=split_ds["train"],
    eval_dataset=split_ds["test"],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print("Iniciando búsqueda de hiperparámetros...")
best_run = trainer_hpo.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=my_hp_space,
    n_trials=5 
)

# Validación cruzada
print(f"\nMejores parámetros: {best_run.hyperparameters}")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

labels = np.array(tokenized_ds["label"])

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n---VALIDANDO FOLD {fold + 1} ---")
    
    final_args = TrainingArguments(
        output_dir=f"./fold_{fold}",
        learning_rate=best_run.hyperparameters["learning_rate"],
        num_train_epochs=best_run.hyperparameters["num_train_epochs"],
        per_device_train_batch_size=best_run.hyperparameters["per_device_train_batch_size"],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        warmup_ratio=0.1,
        logging_steps=20,
        fp16=True if torch.cuda.is_available() else False
    )

    trainer_cv = Trainer(
        model_init=model_init,
        args=final_args,
        train_dataset=tokenized_ds.select(train_idx),
        eval_dataset=tokenized_ds.select(val_idx),
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )
    
    trainer_cv.train()
    eval_res = trainer_cv.evaluate()
    cv_scores.append(eval_res["eval_f1"])
    print(f"Fold {fold+1} F1: {eval_res['eval_f1']:.4f}")

print(f"\nRESULTADO FINAL CV: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1320.88it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream 

Iniciando búsqueda de hiperparámetros...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1391.69it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1
1,0.485893,0.483254,0.761136
2,0.465917,0.457712,0.779150


[I 2026-03-04 20:05:45,341] Trial 0 finished with value: 0.77915043219033 and parameters: {'learning_rate': 0.00041400294154140545, 'num_train_epochs': 2, 'per_device_train_batch_size': 32}. Best is trial 0 with value: 0.77915043219033.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1265.78it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not o

...

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Fold 4 F1: 0.7788

---VALIDANDO FOLD 5 ---


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1265.02it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1241.59it/s, Materiali

Epoch,Training Loss,Validation Loss,F1
1,0.507136,0.498540,0.754528
2,0.466642,0.480664,0.765984


Fold 5 F1: 0.7660

RESULTADO FINAL CV: 0.7716 (+/- 0.0115)


In [8]:
import json
import torch
from transformers import DataCollatorWithPadding

print("Reentrenando modelo final con el dataset completo...")

final_training_args = TrainingArguments(
    output_dir="./modelo_final_exist",
    learning_rate=best_run.hyperparameters["learning_rate"],
    num_train_epochs=best_run.hyperparameters["num_train_epochs"],
    per_device_train_batch_size=best_run.hyperparameters["per_device_train_batch_size"],
    warmup_ratio=0.1,
    fp16=True if torch.cuda.is_available() else False,
    save_strategy="no" 
)

trainer_final = Trainer(
    model_init=model_init,
    args=final_training_args,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)

trainer_final.train()

ruta_test = 'dataset_task1_exist2025/test.json'
with open(ruta_test, 'r', encoding='utf-8') as f:
    test_data = json.load(f)

ids_test = []
textos_test = []
for key, value in test_data.items():
    ids_test.append(value['id_EXIST'])
    textos_test.append(value['tweet'])

print("Generando predicciones para el test oficial...")
model_final = trainer_final.model
model_final.eval()
model_final.to("cuda" if torch.cuda.is_available() else "cpu")

predicciones_raw = []

batch_size = 16
for i in range(0, len(textos_test), batch_size):
    batch_texts = textos_test[i:i+batch_size]
    inputs = tokenizer(batch_texts, truncation=True, padding=True, max_length=128, return_tensors="pt").to(model_final.device)
    
    with torch.no_grad():
        outputs = model_final(**inputs)
        preds = torch.argmax(outputs.logits, dim=-1)
        predicciones_raw.extend(preds.cpu().tolist())

label_map_inv = {1: "YES", 0: "NO"}
output_json = []

for id_ev, pred_idx in zip(ids_test, predicciones_raw):
    output_json.append({
        'test_case': 'EXIST2025',
        'id': id_ev,
        'value': label_map_inv[pred_idx]
    })

# Guardar archivo
nombre_archivo = 'Thinkpol_modelo_DISTILBERT_LoRA.json'
with open(nombre_archivo, 'w', encoding='utf-8') as f:
    json.dump(output_json, f, ensure_ascii=False, indent=4)

print(f"Proceso finalizado. Archivo {nombre_archivo} listo para entrega.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Reentrenando modelo final con el dataset completo...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1299.02it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1255.97it/s, Materiali

Step,Training Loss


Generando predicciones para el test oficial...
Proceso finalizado. Archivo Thinkpol_modelo_DISTILBERT_LoRA.json listo para entrega.
